# (Opcjonalnie) Implementacja prostej sieci neuronowej od podstaw

**Plan notatnika:**
1. Dane i architektura
2. Komponenty sieci
3. Kontrola gradientów
4. Budowa i trening modelu
5. Analiza wyników i predykcje dla nowych punktów
6. Podsumowanie

**Cel:** zrozumieć mechanizmy działania sieci neuronowej, budując prosty model regresji wyłącznie w bibliotece **NumPy**. Krok po kroku zaimplementujemy propagację w przód, obliczanie błędu, propagację wsteczną (*backpropagation*) i proces uczenia.

**Uwaga:** notatnik ma charakter edukacyjny. W praktyce sieci buduje się we frameworkach takich jak Keras/TensorFlow, które są zoptymalizowane i liczą gradienty automatycznie (notatnik 03).

## 1. Dane i architektura

Użyjemy małego zbioru regresyjnego z dwiema cechami wejściowymi. Zmienna docelowa to dokładna funkcja liniowa $y = 0{,}8\,x_1 + 1{,}2\,x_2 + 0{,}3$ (bez szumu), więc sieć ma szansę nauczyć się jej niemal idealnie. Kilka obserwacji (zamiast jednej) pozwala sprawdzić, czy model naprawdę się uczy.

Architektura:
- warstwa wejściowa: 2 neurony (2 cechy),
- warstwa ukryta: 2 neurony z aktywacją sigmoid,
- warstwa wyjściowa: 1 neuron bez aktywacji (regresja).

Liczba parametrów: $(2+1)\cdot2 + (2+1)\cdot1 = 9$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

X = np.array([
    [1.4, 0.7],
    [0.2, 1.3],
    [2.0, 1.0],
    [1.0, 2.0],
    [0.5, 0.4],
    [1.8, 1.5],
    [2.2, 0.3],
    [0.8, 1.7],
], dtype=float)

y_true = 0.8 * X[:, [0]] + 1.2 * X[:, [1]] + 0.3

print(f'X shape: {X.shape}')
print(f'y_true shape: {y_true.shape}')

## 2. Komponenty sieci

Każdy etap implementujemy osobno, żeby było jasne, co dzieje się w przejściu w przód (*forward*) i wstecz (*backward*).

**Funkcja aktywacji** - sigmoid i jej pochodna (notatnik 02):

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

**Inicjalizacja** wag losowymi, małymi wartościami (mniejsza wariancja na starcie stabilizuje uczenie z sigmoidą), a biasów zerami. Wszystkie wagi nie mogą być równe, bo wtedy neurony w warstwie uczyłyby się identycznie.

In [ ]:
def initialize_parameters(n_x, n_h, n_y):
    W1 = np.random.randn(n_x, n_h) * np.sqrt(1.0 / n_x)
    b1 = np.zeros((1, n_h))
    W2 = np.random.randn(n_h, n_y) * np.sqrt(1.0 / n_h)
    b2 = np.zeros((1, n_y))
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

**Propagacja w przód**: $Z_1 = XW_1 + b_1$, $A_1 = \sigma(Z_1)$, $Z_2 = A_1W_2 + b_2$. W regresji wyjście to po prostu $A_2 = Z_2$ (bez aktywacji). Zmienne pośrednie zapamiętujemy w `cache`, bo będą potrzebne w przejściu wstecz.

In [ ]:
def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters['W1'], parameters['b1'], parameters['W2'], parameters['b2']

    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = Z2

    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}
    return A2, cache

**Funkcja straty** - błąd średniokwadratowy $L = \frac{1}{m}\sum (A_2 - Y)^2$:

In [ ]:
def compute_loss(A2, Y):
    m = Y.shape[0]
    return np.sum((A2 - Y) ** 2) / m

**Propagacja wsteczna** to reguła łańcuchowa stosowana od wyjścia do wejścia:

$$dZ_2 = \frac{2}{m}(A_2 - Y),\quad dW_2 = A_1^T dZ_2,\quad db_2 = \sum dZ_2$$
$$dA_1 = dZ_2 W_2^T,\quad dZ_1 = dA_1 \odot \sigma'(Z_1),\quad dW_1 = X^T dZ_1,\quad db_1 = \sum dZ_1$$

Wartość $\frac{2}{m}(A_2-Y)$ to pochodna MSE po wyjściu sieci; $\sigma'$ to pochodna sigmoidy; $\odot$ oznacza mnożenie element po elemencie.

In [ ]:
def backward_propagation(parameters, cache, X, Y):
    m = X.shape[0]
    W2 = parameters['W2']
    A1, A2, Z1 = cache['A1'], cache['A2'], cache['Z1']

    dZ2 = 2 * (A2 - Y) / m
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * sigmoid_derivative(Z1)
    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

**Aktualizacja parametrów** - krok spadku wzdłuż gradientu (notatnik 03): $\theta \leftarrow \theta - \alpha\, d\theta$.

In [ ]:
def update_parameters(parameters, grads, learning_rate):
    return {name: parameters[name] - learning_rate * grads['d' + name] for name in parameters}

## 3. Kontrola gradientów

Błędy w propagacji wstecznej są łatwe do popełnienia, a trudne do zauważenia (sieć uczy się gorzej, ale nie zgłasza błędu). Standardową kontrolą jest **porównanie z gradientem numerycznym**: pochodną przybliżamy różnicą centralną

$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + \varepsilon) - L(\theta - \varepsilon)}{2\varepsilon}$$

dla każdego parametru z osobna. Jeśli implementacja jest poprawna, wyniki prawie się pokrywają.

In [ ]:
def numerical_gradients(parameters, X, Y, eps=1e-6):
    numeric = {}
    for name, p in parameters.items():
        grad = np.zeros_like(p)
        for idx in np.ndindex(p.shape):
            original = p[idx]
            p[idx] = original + eps
            loss_plus = compute_loss(forward_propagation(X, parameters)[0], Y)
            p[idx] = original - eps
            loss_minus = compute_loss(forward_propagation(X, parameters)[0], Y)
            p[idx] = original
            grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        numeric['d' + name] = grad
    return numeric

check_params = initialize_parameters(2, 2, 1)
_, check_cache = forward_propagation(X, check_params)
analytic = backward_propagation(check_params, check_cache, X, y_true)
numeric = numerical_gradients(check_params, X, y_true)

for name in analytic:
    diff = np.max(np.abs(analytic[name] - numeric[name]))
    print(f'{name}: największa różnica między gradientem analitycznym a numerycznym = {diff:.2e}')

Różnice są rzędu $10^{-9}$, czyli propagacja wsteczna działa poprawnie. Tej techniki używa się do testowania własnych implementacji (frameworki nie wymagają jej, bo liczą gradienty automatycznie).

## 4. Budowa i trening modelu

Łączymy komponenty w pętlę treningową: w każdej iteracji przejście w przód, strata, przejście wstecz i aktualizacja wag. Ponieważ mamy tylko 8 obserwacji, w każdej iteracji używamy ich wszystkich (*batch gradient descent*, notatnik 03).

In [ ]:
def build_model(X, Y, n_h, num_iterations=3000, learning_rate=0.1, print_loss=False):
    n_x = X.shape[1]
    n_y = Y.shape[1]

    parameters = initialize_parameters(n_x, n_h, n_y)
    losses = []

    for i in range(num_iterations):
        A2, cache = forward_propagation(X, parameters)
        loss = compute_loss(A2, Y)
        losses.append(loss)

        grads = backward_propagation(parameters, cache, X, Y)
        parameters = update_parameters(parameters, grads, learning_rate)

        if print_loss and i % 300 == 0:
            print(f'Iteracja {i:4d}: loss = {loss:.6f}')

    return {'parameters': parameters, 'losses': losses}

np.random.seed(SEED)
model = build_model(X, y_true, n_h=2, num_iterations=3000, learning_rate=0.1, print_loss=True)

## 5. Analiza wyników i predykcje dla nowych punktów

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model['losses'])
plt.yscale('log')
plt.title('Krzywa uczenia (MSE, skala logarytmiczna)')
plt.xlabel('Iteracja')
plt.ylabel('MSE')
plt.grid(True)
plt.show()

### Jak interpretować krzywą uczenia
- Jeżeli MSE systematycznie maleje, model uczy się poprawnie.
- Jeżeli MSE zatrzymuje się wysoko, model ma zbyt małą pojemność albo współczynnik uczenia jest nieodpowiedni.
- Jeżeli MSE oscyluje lub rośnie, współczynnik uczenia bywa za duży.

In [ ]:
y_pred, _ = forward_propagation(X, model['parameters'])
print(f'Końcowy MSE: {compute_loss(y_pred, y_true):.6f}')

print('\nPorównanie kilku predykcji (dane treningowe):')
for i in range(5):
    print(f'Próbka {i}: y_true = {y_true[i, 0]:.3f}, y_pred = {y_pred[i, 0]:.3f}')

Uczyliśmy i oceniliśmy model na tych samych 8 obserwacjach, więc niski błąd jest oczekiwany. Sprawdźmy więc, jak sieć radzi sobie z **nowymi punktami**: dwoma leżącymi w zakresie danych treningowych (cechy od 0,2 do 2,2) i jednym **poza** tym zakresem.

In [ ]:
X_new = np.array([[1.5, 1.0], [0.6, 1.9], [3.0, 2.5]])
y_new_true = 0.8 * X_new[:, [0]] + 1.2 * X_new[:, [1]] + 0.3
y_new_pred, _ = forward_propagation(X_new, model['parameters'])

for point, t, p in zip(X_new, y_new_true[:, 0], y_new_pred[:, 0]):
    print(f'x = {point}:  y_true = {t:.3f}, y_pred = {p:.3f}, błąd = {abs(t - p):.3f}')

Dla punktów leżących w zakresie danych treningowych błąd jest rzędu 0,02, czyli sieć faktycznie nauczyła się zależności, a nie zapamiętała ośmiu przykładów. Dla punktu **spoza zakresu** błąd jest ponad pięćdziesiąt razy większy: sieć zaniża wynik. Neurony ukryte z sigmoidą się nasycają (ich wyjście nie przekracza 1), a nasza sieć ma tylko dwa takie neurony, więc poza obszarem uczenia nie potrafi dobrze **ekstrapolować**. Modele uczone na danych z jednego zakresu nie powinny być stosowane do obserwacji znacznie spoza niego.

## 6. Podsumowanie

W tym notatniku zbudowaliśmy od zera prostą sieć neuronową w NumPy i prześledziliśmy pełny cykl uczenia:
1. przejście w przód (forward pass),
2. obliczenie funkcji straty (MSE),
3. przejście wstecz (gradienty z reguły łańcuchowej),
4. aktualizacja wag (spadek wzdłuż gradientu).

Dodatkowo sprawdziliśmy gradienty **metodą numeryczną**, co jest standardową kontrolą poprawności własnej implementacji. Równania z teorii przekładają się bezpośrednio na kod, a każda iteracja obniża błąd modelu. W praktyce używamy Keras/TensorFlow, ale zrozumienie „ręcznej” implementacji ułatwia debugowanie i strojenie większych modeli.